<a href="https://colab.research.google.com/github/Shacxify/prompt-engineering-exercises/blob/main/02_react_code_generation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Exercise 2 - Code Generation with ReAct Prompting

**Cash Johnson | BUS4 118S - Agentic AI for Business | Prof. Haubrich**

**Tools used:** Google Colab + Google Gemini API (`google-genai` SDK). The setup cell picks
the model at runtime from what the key can actually call, and prints which one it landed on,
because free-tier quota is metered per model per day and model names get retired.
No agent framework. Autogen or LangChain would run this loop for me, and the loop is the thing being
graded, so it is written out by hand: the OBSERVATION is literally the stdout of a test harness in
the cell above it.

**Goal:** generate working Python through an explicit ReAct cycle - **reason -> act -> observe -> fix** -
instead of asking for code once and hoping. The observation step is real: the notebook executes the
generated module against a hidden acceptance suite and feeds the actual traceback back to the model.

**Task being generated:** the consignor payout calculator for VNTG OS. Given a list of sales, work out
what each consignor is owed. Real money, so rounding and exclusions have to be right.

---

### The loop

```
THOUGHT   -> what the model knows and what it will change
PLAN      -> numbered implementation steps
ACTION    -> one complete python module in a single code block
    |
    v  (the notebook actually runs it)
OBSERVATION -> real pass/fail lines and real tracebacks from the harness
    |
    +----> back to THOUGHT, up to 3 attempts, stop early on all-pass
```

### A note on the spec, since it matters for grading

The spec below is **deliberately complete on the interface** (inputs, output shape, business rules,
allowed libraries, return format) and **deliberately quiet on exact edge-case behavior** - it says
"fail loudly and specifically" rather than naming the exception, and "exact to the cent" rather than
naming the rounding mode. That is the point of ReAct. If the first prompt contains every answer there
is nothing for the observe step to teach, and the loop is theater. The acceptance suite is the ground
truth the model does not get to see, exactly like a real ticket with real tests behind it.

### Techniques from the module used in this notebook

| Module technique | Where it shows up here |
|---|---|
| **ReACT (reasoning + acting)** | the loop itself: THOUGHT and PLAN are the reasoning, ACTION is the acting, and the harness supplies a real OBSERVATION |
| **Chain-of-thought** | the THOUGHT stage, required before any code is written |
| **RSIP (recursive self-improvement)** | each pass evaluates the previous response against requirements, applies corrections, and generates a new version |
| **System prompt vs user prompt** | the stage contract lives in the SYSTEM prompt, the task spec and every OBSERVATION are USER prompts |
| **Role-based prompting** | "You are a Python engineer working inside an explicit ReAct loop" |
| **Clarity and specificity** | the spec names inputs, types, output shape, business rules, allowed libraries, and formatting |
| **Content structuring** | four named sections in a fixed order, which is also what makes the code extractable by regex |
| **Negative prompting** | "never a diff", "no prose outside the four sections", "never state that a test passes unless the OBSERVATION told you it passed" |
| **Iteration and experimentation** | the loop runs until green or three attempts, and section 7 reports what each pass actually caught |
| **Learning from failed prompts** | the notes in section 7 trace each failure back to what the prompt left unsaid |
| **Temperature** | 0.2, low enough to be near-deterministic, high enough to let attempt 2 move off a wrong approach |
| **AI laziness / hallucination guards** | the model is explicitly barred from claiming a passing test, because self-reported success is the failure mode this exercise exists to prevent |

In [1]:
!pip install -q -U google-genai

from google import genai
from google.genai import types
from google.genai import errors as genai_errors
import json, time, re

try:
    from google.colab import userdata
    API_KEY = userdata.get('GOOGLE_API_KEY')
except Exception:
    import getpass
    API_KEY = getpass.getpass('Gemini API key: ')

client = genai.Client(api_key=API_KEY)

# ---------------------------------------------------------------------------
# Free-tier quota is metered PER PROJECT, PER MODEL, PER DAY. So each of the
# three notebooks pins a DIFFERENT model and gets its own daily allowance
# instead of all three draining one bucket. This notebook takes slot 1.
#
# Model names change and not every listed model is callable on every key, so
# rather than hard-coding one, list what the key can see and probe until one
# actually answers. Set PIN below to override.
# ---------------------------------------------------------------------------
PIN = None          # e.g. 'gemini-3.6-flash' to force a specific model
SLOT = 1       # which model in the preference order this notebook claims

def _candidates():
    names = []
    for m in client.models.list():
        actions = getattr(m, 'supported_actions', None) or []
        if actions and 'generateContent' not in actions:
            continue
        n = m.name.replace('models/', '')
        if 'embedding' in n or 'imagen' in n or 'veo' in n or 'tts' in n:
            continue
        if 'flash' in n or 'pro' in n:
            names.append(n)
    names.sort(key=lambda n: (0 if 'flash-lite' in n else 1 if 'flash' in n else 2, n))
    return names

def _probe(name):
    try:
        r = client.models.generate_content(
            model=name, contents='say ok',
            config=types.GenerateContentConfig(temperature=0))
        return bool((r.text or '').strip())
    except genai_errors.APIError as e:
        print(f'  {name}: unavailable ({getattr(e, "code", "?")})')
        return False

MODEL = None
if PIN:
    MODEL, pool = PIN, [PIN]
else:
    pool = _candidates()
    print(f'{len(pool)} candidate models on this key')
    for name in pool[SLOT:] + pool[:SLOT]:
        if _probe(name):
            MODEL = name
            break
assert MODEL, f'no callable model found. Candidates were: {pool[:8]}'
print('Using:', MODEL)

CALLS = 0
RETRY_ON = {500, 502, 503, 504}

# Free tier meters TWO different quotas and they need opposite responses:
#   per minute (RPM 5-10) -> transient, wait it out and carry on
#   per day    (RPD 20)   -> gone until reset, no amount of waiting helps
# The 429 body names which one via quotaId, so read it instead of guessing.
MIN_INTERVAL = 13.0     # seconds between calls, keeps us under ~5 RPM
_last_call = [0.0]

def _pace():
    gap = time.time() - _last_call[0]
    if gap < MIN_INTERVAL:
        time.sleep(MIN_INTERVAL - gap)
    _last_call[0] = time.time()

def _quota_kind(err):
    blob = str(getattr(err, 'message', '') or err)
    if 'PerDay' in blob:
        return 'day'
    if 'PerMinute' in blob:
        return 'minute'
    return 'unknown'

def _retry_delay(err, default=30.0):
    m = re.search(r"retryDelay'?\s*:\s*'?(\d+(?:\.\d+)?)s", str(err))
    return float(m.group(1)) + 2 if m else default

def _call(model, contents, cfg, retries=4):
    """Every API call goes through here so the run is paced, counted, and survives a blip."""
    global CALLS
    for attempt in range(retries):
        _pace()
        try:
            r = client.models.generate_content(model=model, contents=contents, config=cfg)
            CALLS += 1
            return r
        except genai_errors.APIError as e:
            code = getattr(e, 'code', None)
            if code == 429:
                kind = _quota_kind(e)
                if kind == 'day':
                    raise RuntimeError(
                        f'Daily quota gone for {model} after {CALLS} calls this session. '
                        f'The cap is per model per day, so set PIN to another model and re-run '
                        f'from this cell, or wait for reset. See https://ai.dev/rate-limit') from e
                if attempt == retries - 1:
                    raise
                wait = _retry_delay(e)
                print(f'  [429 {kind}] over the per-minute limit, waiting {wait:.0f}s')
                time.sleep(wait)
                continue
            if code not in RETRY_ON or attempt == retries - 1:
                raise
            wait = 3 * 2 ** attempt
            print(f'  [{code}] model busy, retrying in {wait}s')
            time.sleep(wait)

def ask(user_prompt, system=None, temperature=0.2, model=None):
    kwargs = {'temperature': temperature}
    if system is not None:
        kwargs['system_instruction'] = system
    resp = _call(model or MODEL, user_prompt, types.GenerateContentConfig(**kwargs))
    return (resp.text or '').strip()


print('helper ready | model:', MODEL, '| calls so far:', CALLS)

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.5/56.5 kB 2.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 10.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 262.4/262.4 kB 6.3 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires google-auth==2.49.0, but you have google-auth 2.58.0 which is incompatible.


27 candidate models on this key
Using: gemini-3.1-flash-lite
helper ready | model: gemini-3.1-flash-lite | calls so far: 0


## 1. The ReAct SYSTEM prompt (stage definitions)

This is the SYSTEM prompt: role, stage contract, and hard rules. It is set once and never changes.
The task spec and every OBSERVATION below are USER prompts. Stages are separated by name so the
reasoning is auditable, the THOUGHT stage is chain-of-thought made mandatory, and the fixed section
order is what lets the code be extracted mechanically. `temperature=0.2` because code generation wants near-determinism but not zero, and
the model has to be able to move off a wrong approach on attempt 2.

## 2. The task spec (the USER prompt)

| Constraint type | Specified |
|---|---|
| Inputs | list of dicts, every key and type named |
| Output | exact dict shape, exact value types |
| Business rules | payable window, split, tier threshold |
| Libraries | stdlib only, `decimal` and `typing` named |
| Formatting | single module, type hints, docstring, one fenced block |
| Error handling | required, mode left to the model on purpose |

In [2]:
REACT_SYSTEM = """You're a Python engineer working a reason, act, observe, fix loop.

Every turn, give me these four sections in this order, with these headers:

THOUGHT:
  What you know, what the last observation changed about that, and what you're doing
  differently this time. No code in here.

PLAN:
  Numbered steps, keep it to five.

ACTION:
  One ```python block with the whole module in it. Not a diff, not a fragment, not two
  blocks. Rewrite the whole thing every turn.

EXPECTATION:
  What you think the harness will say, for anything you're genuinely unsure about.

The OBSERVATION you get back is real output from a real test harness that already ran
your code. Take it at face value, don't argue with it, and don't tell me a test passed
unless it told you it passed. Standard library only, nothing third party."""

print(REACT_SYSTEM)

TASK = """Write the consignor payout calculator for VNTG OS.

  calculate_payouts(sales: list[dict]) -> dict

Each sale in the list looks like this:

  item_id          str    "VN-0184"
  consignor_id     str    "C-207"
  sale_price       str    dollars as a string, "129.99"
  status           str    "sold", "returned", or "pending"
  days_since_sale  int    whole days since the sale closed

How payouts work:
  A sale only counts once it's marked sold and it's at least 7 days old, since buyers
  get 7 days to return.
  The consignor takes 60% of the sale price, or 65% if the item went for $200 or more.
  Money is decimal.Decimal and has to be exact to the cent.

Hand back a dict keyed by consignor_id. Each value holds item_count, gross_sales, and
payout, both money values quantized to 2 places. Leave out consignors with nothing
payable.

Standard library only, no pandas or numpy. One function called calculate_payouts plus
whatever private helpers you want, type hints on the signatures, a docstring on the
public one. Code only, in a single ```python block.

Bad input should fail loudly and specifically rather than quietly returning a wrong
number."""

print(TASK)

You're a Python engineer working a reason, act, observe, fix loop.

Every turn, give me these four sections in this order, with these headers:

THOUGHT:
  What you know, what the last observation changed about that, and what you're doing
  differently this time. No code in here.

PLAN:
  Numbered steps, keep it to five.

ACTION:
  One ```python block with the whole module in it. Not a diff, not a fragment, not two
  blocks. Rewrite the whole thing every turn.

EXPECTATION:
  What you think the harness will say, for anything you're genuinely unsure about.

The OBSERVATION you get back is real output from a real test harness that already ran
your code. Take it at face value, don't argue with it, and don't tell me a test passed
unless it told you it passed. Standard library only, nothing third party.
Write the consignor payout calculator for VNTG OS.

  calculate_payouts(sales: list[dict]) -> dict

Each sale in the list looks like this:

  item_id          str    "VN-0184"
  consignor_id 

## 3. The acceptance suite (the OBSERVATION source)

The model never sees this. It only ever sees what the harness prints.

| Test | What it is actually checking |
|---|---|
| `empty_input` | no crash on an empty sales list |
| `basic_split` | the plain 60% path |
| `return_window` | day 3 is not payable, day 7 is |
| `returned_excluded` | a return never pays out |
| `tier_boundary` | exactly $200.00 gets 65%, the threshold is inclusive |
| `rounding_half_up` | $200.10 at 65% is 130.065 exactly - half-up gives 130.07, banker's rounding and floats give 130.06 |
| `negative_price` | raises `ValueError`, not a silent negative payout |
| `two_consignors` | aggregation and item counts |
| `returns_decimal` | values are `Decimal`, not `float` |
| `unknown_status` | a status outside the three named ones raises `ValueError` rather than being skipped |
| `unparseable_price` | a junk price string raises `ValueError`, not a raw `decimal.InvalidOperation` |

In [3]:
from decimal import Decimal
import traceback, re

def _sale(item, consignor, price, status='sold', days=30):
    return {'item_id': item, 'consignor_id': consignor, 'sale_price': price,
            'status': status, 'days_since_sale': days}

def t_empty_input(f):
    assert f([]) == {}, f'expected {{}} for no sales, got {f([])!r}'

def t_basic_split(f):
    out = f([_sale('VN-1', 'C-1', '100.00')])
    assert out['C-1']['payout'] == Decimal('60.00'), out['C-1']['payout']
    assert out['C-1']['gross_sales'] == Decimal('100.00'), out['C-1']['gross_sales']
    assert out['C-1']['item_count'] == 1, out['C-1']['item_count']

def t_return_window(f):
    day3 = f([_sale('VN-2', 'C-1', '80.00', days=3)])
    assert day3 == {}, f'day 3 is inside the return window, should not pay: {day3!r}'
    day7 = f([_sale('VN-3', 'C-1', '80.00', days=7)])
    assert day7['C-1']['payout'] == Decimal('48.00'), day7['C-1']['payout']

def t_returned_excluded(f):
    out = f([_sale('VN-4', 'C-1', '90.00', status='returned')])
    assert out == {}, f'a returned item must never pay out: {out!r}'

def t_tier_boundary(f):
    out = f([_sale('VN-5', 'C-2', '200.00')])
    assert out['C-2']['payout'] == Decimal('130.00'), \
        f"$200.00 is the 65% tier (inclusive): expected 130.00, got {out['C-2']['payout']}"

def t_rounding_half_up(f):
    out = f([_sale('VN-6', 'C-2', '200.10')])
    assert out['C-2']['payout'] == Decimal('130.07'), \
        f"200.10 * 0.65 = 130.065 exactly, round half up to 130.07, got {out['C-2']['payout']}"

def t_negative_price(f):
    try:
        f([_sale('VN-7', 'C-3', '-20.00')])
    except ValueError:
        return
    except Exception as e:
        raise AssertionError(f'expected ValueError on a negative price, got {type(e).__name__}')
    raise AssertionError('a negative sale price was accepted silently')

def t_two_consignors(f):
    out = f([_sale('VN-8', 'C-1', '50.00'), _sale('VN-9', 'C-1', '25.00'),
             _sale('VN-10', 'C-4', '300.00'), _sale('VN-11', 'C-4', '10.00', days=1)])
    assert out['C-1']['item_count'] == 2, out['C-1']['item_count']
    assert out['C-1']['payout'] == Decimal('45.00'), out['C-1']['payout']
    assert out['C-4']['item_count'] == 1, 'the 1-day-old sale is not payable yet'
    assert out['C-4']['payout'] == Decimal('195.00'), out['C-4']['payout']

def t_returns_decimal(f):
    out = f([_sale('VN-12', 'C-1', '100.00')])
    assert isinstance(out['C-1']['payout'], Decimal), \
        f"payout must be Decimal, got {type(out['C-1']['payout']).__name__}"

def t_unknown_status(f):
    try:
        f([_sale('VN-13', 'C-5', '75.00', status='refunded')])
    except ValueError:
        return
    except Exception as e:
        raise AssertionError(f'expected ValueError on an unrecognised status, got {type(e).__name__}')
    raise AssertionError('an unrecognised status was accepted silently')

def t_unparseable_price(f):
    try:
        f([_sale('VN-14', 'C-5', 'N/A')])
    except ValueError:
        return
    except Exception as e:
        raise AssertionError(f'a bad price string should raise ValueError, not {type(e).__name__}')
    raise AssertionError('an unparseable sale_price was accepted silently')

TESTS = [
    ('empty_input', t_empty_input),
    ('basic_split', t_basic_split),
    ('return_window', t_return_window),
    ('returned_excluded', t_returned_excluded),
    ('tier_boundary', t_tier_boundary),
    ('rounding_half_up', t_rounding_half_up),
    ('negative_price', t_negative_price),
    ('two_consignors', t_two_consignors),
    ('returns_decimal', t_returns_decimal),
    ('unknown_status', t_unknown_status),
    ('unparseable_price', t_unparseable_price),
]
print(len(TESTS), 'acceptance tests loaded')

def extract_code(reply: str) -> str:
    """Pull the single python block out of an ACTION section."""
    blocks = re.findall(r'```(?:python)?\n(.*?)```', reply, re.DOTALL)
    if not blocks:
        raise ValueError('no fenced python block found in the reply')
    return blocks[0].strip()

def run_acceptance(code_str: str):
    """Execute the generated module and run every test. Returns (all_passed, report)."""
    ns = {}
    try:
        exec(code_str, ns)
    except Exception:
        return False, 'MODULE FAILED TO EXECUTE:\n' + traceback.format_exc(limit=2)
    if 'calculate_payouts' not in ns:
        return False, 'No function named calculate_payouts was defined.'
    fn = ns['calculate_payouts']
    lines, passed = [], True
    for name, test in TESTS:
        try:
            test(fn)
            lines.append(f'PASS   {name}')
        except AssertionError as e:
            passed = False
            lines.append(f'FAIL   {name}: {e}')
        except Exception as e:
            passed = False
            lines.append(f'ERROR  {name}: {type(e).__name__}: {e}')
    score = sum(l.startswith('PASS') for l in lines)
    lines.append(f'--- {score}/{len(TESTS)} passing ---')
    return passed, '\n'.join(lines)

print('harness ready')

11 acceptance tests loaded
harness ready


## 4. Run the ReAct loop

Attempt 1 gets the task spec. Every later attempt gets the real harness output as its OBSERVATION
and nothing else new. The loop breaks the moment everything passes, so the transcript below is the
honest number of cycles it took, not a scripted two.

In [4]:
transcript = []
conversation = TASK
final_code = None
MAX_ATTEMPTS = 3

for attempt in range(1, MAX_ATTEMPTS + 1):
    print('#' * 72)
    print(f'# ATTEMPT {attempt}')
    print('#' * 72)

    reply = ask(conversation, system=REACT_SYSTEM)
    print(reply)

    generated = extract_code(reply)
    passed, report = run_acceptance(generated)

    print('\n' + '-' * 72)
    print('OBSERVATION (real harness output)')
    print('-' * 72)
    print(report)
    print()

    transcript.append({'attempt': attempt, 'reply': reply, 'code': generated,
                       'passed': passed, 'report': report})
    final_code = generated

    if passed:
        print(f'All tests green on attempt {attempt}. Stopping the loop.')
        break

    conversation = f"""OBSERVATION from the test harness on your previous ACTION:

{report}

The failing cases above are the real behavior of your code. Diagnose each failure specifically in
THOUGHT, then output the complete corrected module in ACTION. Do not change anything that is
already passing.

The original task, unchanged:
{TASK}"""
else:
    print(f'Hit the {MAX_ATTEMPTS}-attempt ceiling.')

########################################################################
# ATTEMPT 1
########################################################################
THOUGHT:
The requirements specify using `decimal.Decimal` for financial accuracy. I need to filter sales based on the "sold" status and the 7-day return window. The commission logic is a simple conditional (60% vs 65% for >= $200). I must ensure the output dictionary only includes consignors with at least one valid sale. I will use `decimal.ROUND_HALF_UP` for currency quantization.

PLAN:
1. Import `Decimal` and `ROUND_HALF_UP` from `decimal`.
2. Iterate through the sales list, validating keys and types.
3. Filter for `status == "sold"` and `days_since_sale >= 7`.
4. Calculate commission based on the $200 threshold.
5. Aggregate results into a dictionary, quantizing values to two decimal places.

ACTION:
```python
from decimal import Decimal, ROUND_HALF_UP

def calculate_payouts(sales: list[dict]) -> dict:
    """
    Calculates c

In [5]:
# =====================================================================
# FINAL MODULE, clean re-run of the suite, and a run on real-shaped data
# =====================================================================

print(final_code)

# Re-run the suite against the final module, cleanly, so the result is visible on its own.
passed, report = run_acceptance(final_code)
print(report)
print('\nALL TESTS PASSING' if passed else '\nSTILL FAILING')

ns = {}
exec(final_code, ns)
calculate_payouts = ns['calculate_payouts']

SEPTEMBER_SALES = [
    {'item_id': 'VN-4401', 'consignor_id': 'C-207', 'sale_price': '245.00', 'status': 'sold',     'days_since_sale': 21},
    {'item_id': 'VN-4402', 'consignor_id': 'C-207', 'sale_price': '68.00',  'status': 'sold',     'days_since_sale': 14},
    {'item_id': 'VN-4403', 'consignor_id': 'C-207', 'sale_price': '112.50', 'status': 'returned', 'days_since_sale': 19},
    {'item_id': 'VN-4404', 'consignor_id': 'C-118', 'sale_price': '200.10', 'status': 'sold',     'days_since_sale': 9},
    {'item_id': 'VN-4405', 'consignor_id': 'C-118', 'sale_price': '34.00',  'status': 'sold',     'days_since_sale': 2},
    {'item_id': 'VN-4406', 'consignor_id': 'C-330', 'sale_price': '89.99',  'status': 'pending',  'days_since_sale': 0},
    {'item_id': 'VN-4407', 'consignor_id': 'C-412', 'sale_price': '1250.00','status': 'sold',     'days_since_sale': 40},
]

payouts = calculate_payouts(SEPTEMBER_SALES)

print(f"{'CONSIGNOR':<12}{'ITEMS':>7}{'GROSS':>12}{'PAYOUT':>12}")
print('-' * 43)
for cid in sorted(payouts):
    row = payouts[cid]
    print(f"{cid:<12}{row['item_count']:>7}{'$' + str(row['gross_sales']):>12}{'$' + str(row['payout']):>12}")
print('-' * 43)
print(f"{'TOTAL':<12}{sum(r['item_count'] for r in payouts.values()):>7}"
      f"{'$' + str(sum(r['gross_sales'] for r in payouts.values())):>12}"
      f"{'$' + str(sum(r['payout'] for r in payouts.values())):>12}")
print('\nC-330 is absent because that sale is still pending. Correct.')

for t in transcript:
    score = t['report'].strip().splitlines()[-1]
    fails = [l for l in t['report'].splitlines() if l.startswith(('FAIL', 'ERROR'))]
    print(f"Attempt {t['attempt']}: {score}")
    for f in fails:
        print('   ', f)
    print()

from decimal import Decimal, ROUND_HALF_UP

def calculate_payouts(sales: list[dict]) -> dict:
    """
    Calculates consignor payouts based on sales data.
    """
    payouts = {}

    for sale in sales:
        status = sale.get("status")
        if status not in ("sold", "returned", "pending"):
            raise ValueError(f"Unknown status: {status}")

        try:
            price = Decimal(sale["sale_price"])
        except Exception:
            raise ValueError(f"Invalid price: {sale['sale_price']}")

        if price < 0:
            raise ValueError("Negative sale price")

        if status != "sold" or sale["days_since_sale"] < 7:
            continue

        consignor_id = sale["consignor_id"]
        
        rate = Decimal("0.65") if price >= Decimal("200") else Decimal("0.60")
        payout_amount = (price * rate).quantize(Decimal("0.01"), rounding=ROUND_HALF_UP)

        if consignor_id not in payouts:
            payouts[consignor_id] = {
                "item_count"

### Notes on fixes and iteration

Fill these in from the actual run above - the failures move a little between runs, and the honest
list is the one the harness printed. The recurring ones across my runs:

1. **`rounding_half_up`** - the usual first-pass miss. `Decimal.quantize()` defaults to
   `ROUND_HALF_EVEN` (banker's rounding), so 130.065 becomes 130.06 and every consignor with a
   half-cent gets shorted a penny. The fix is `quantize(Decimal('0.01'), rounding=ROUND_HALF_UP)`.
   This is the case for making the observe step real: a human reading the code would not catch it,
   and the model does not catch it either until the harness shows it the number.

2. **`negative_price`** - "fail loudly and specifically" is not the same instruction as "raise
   ValueError," and the first pass usually either lets it through or raises something else. The
   OBSERVATION names the expected exception type and attempt 2 pins it.

3. **`tier_boundary`** - `> 200` instead of `>= 200`. Classic off-by-one on an inclusive threshold,
   invisible until a sale lands exactly on it.

**What ReAct bought over a single prompt:** the failures above are all *silent* ones. The code runs,
returns a plausible dollar figure, and is wrong. A one-shot prompt produces that code and nobody
notices until a consignor counts their money. The loop only works because the OBSERVATION is real
program output rather than the model's own opinion of its work.